# 06 - Subcubic theta-family formula discovery

This notebook generates exact Yamada datasets for three subcubic theta-derived spatial-graph families. It is designed for mathematical exploration: compute exact values, fit candidate closed forms outside or inside the notebook, freeze those candidates, and then test them on held-out larger cases.

All three families are built from the same canonical positive two-strand theta skeleton with

$$
q=2m+1
$$

crossings and constituent knot $T(2,q)$.

The target families are:

1. **Cross-linked theta ladder** $\Lambda_m$: every separated upper/lower strand pair is joined by a rung.
2. **Bottom-paired theta family** $P_m^\downarrow$: consecutive lower subdivision vertices are paired non-overlappingly.
3. **Two-sided paired theta family** $P_m^{\updownarrow}$: the same non-overlapping pairing is added on both lower and upper sides.

The non-overlapping pairing is essential: every used subdivision vertex receives exactly one additional incident edge, so

$$
\boxed{\Delta(G)\le 3}
$$

for all three families. In fact $\Lambda_m$ and $P_m^{\updownarrow}$ are cubic, while $P_m^\downarrow$ is subcubic with degree-2 and degree-3 vertices.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import csv
import hashlib
import json
import os
import pickle
import subprocess
import time
from typing import Callable

import networkx as nx
import numpy as np
import sympy as sp

EXPECTED_BRANCH = "integration/arbitrary-knot-fields-final-audit"
CONSTRUCTOR_VERSION = "2026-08-25.three-subcubic-theta-families.safe-return.v3"
RESULT_SCHEMA_VERSION = 1

A = sp.Symbol("A")
SIGMA = A + 1 + A**-1


def _is_repo_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "src" / "knotted_graph").is_dir()
        and (path / "User_guide" / "benchmarks").is_dir()
    )


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if _is_repo_root(candidate):
            return candidate
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )


ROOT = find_repo_root()
RESULTS_DIR = ROOT / "User_guide" / "benchmarks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FULL_CSV = RESULTS_DIR / "06_subcubic_theta_formula_discovery_full.csv"
SHARE_CSV = RESULTS_DIR / "06_subcubic_theta_formula_discovery_share.csv"
HELDOUT_CSV = RESULTS_DIR / "06_subcubic_theta_formula_discovery_heldout.csv"
PROJECTION_CACHE_DIR = RESULTS_DIR / "06_subcubic_theta_formula_discovery_pd_cache"
PROJECTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def git_text(*args: str) -> str:
    proc = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode:
        raise RuntimeError(
            f"git {' '.join(args)} failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc.stdout.strip()


CURRENT_BRANCH = git_text("rev-parse", "--abbrev-ref", "HEAD")
GIT_COMMIT = git_text("rev-parse", "HEAD")

if CURRENT_BRANCH != EXPECTED_BRANCH:
    raise RuntimeError(
        f"This notebook is audited for {EXPECTED_BRANCH!r}, but HEAD is on {CURRENT_BRANCH!r}."
    )

library_status = git_text("status", "--porcelain", "--", "src/knotted_graph")
if library_status:
    raise RuntimeError(
        "The KnottedGraph source tree has uncommitted changes. "
        "Commit or stash them before generating a publication dataset:\n"
        + library_status
    )

import knotted_graph

KG_FILE = Path(knotted_graph.__file__).resolve()
if ROOT not in KG_FILE.parents:
    raise RuntimeError(
        "Python imported knotted_graph from outside this checkout.\n"
        f"Repository root: {ROOT}\nImported package: {KG_FILE}"
    )

from knotted_graph.core.embedding import ensure_embedding
from knotted_graph.projection import PDCode, sample_projections, select_projection
from knotted_graph.invariants.yamada.polynomial import Yamada
from knotted_graph.invariants.yamada.factorized_frontier import (
    build_factorized_frontier,
    native_factorized_available,
    factorized_import_error,
)

if not native_factorized_available():
    raise RuntimeError(
        "The optimized native factorized Yamada backend is unavailable. "
        "Rebuild this checkout before running the dataset.\n"
        f"Import error: {factorized_import_error()!r}"
    )

print("Repository :", ROOT)
print("Branch     :", CURRENT_BRANCH)
print("Commit     :", GIT_COMMIT)
print("Package    :", KG_FILE)
print("Native optimized Yamada backend: AVAILABLE")

## Experiment configuration

In [ ]:
DV_DISCOVERY_N = list(range(3, 11))
DV_HELDOUT_N = list(range(11, 21)) + [50]

# ------------------------------------------------------------
# Main discovery range
# ------------------------------------------------------------
MAX_DISCOVERY_M = 100
MAIN_DISCOVERY_M = list(range(1, MAX_DISCOVERY_M + 1))

# Not computed until formulas are frozen.
MAIN_HELDOUT_M = [101, 125, 150, 200]
COMPUTE_HELDOUT_NOW = False

# ------------------------------------------------------------
# Fast canonical-projection policy
# ------------------------------------------------------------
ROTATION_ORDER = "ZYX"

# These families are DEFINED by the canonical diagram.  Zero rotation is therefore
# the intended projection; no generic projection sampling is necessary.
USE_CANONICAL_ZERO_PROJECTION = True

# Frontier planning repeats part of the Yamada preparation.  It is useful for
# diagnostics but unnecessary for the dataset, so keep it off for maximum speed.
COMPUTE_FRONTIER_DIAGNOSTICS = False

REUSE_PROJECTION_CACHE = True
WRITE_PROJECTION_CACHE = True

# ------------------------------------------------------------
# Correctness / geometry audits
# ------------------------------------------------------------
RUN_UPSTREAM_SANITY = True

# The contact audit is quadratic in the number of PL segments.  Do it only on
# representative members because the geometry is a repeated local motif.
RUN_REPRESENTATIVE_3D_CONTACT_AUDIT = True
REPRESENTATIVE_AUDIT_M = [1, 2, 5, 10]
CONTACT_TOL = 1e-8

# ------------------------------------------------------------
# Runtime / persistence
# ------------------------------------------------------------
RESUME = True
FAIL_FAST = False
NORMALIZE_YAMADA = True

# Rewriting a CSV containing many large exact polynomials after every row becomes
# expensive.  Completed rows remain in memory and are atomically checkpointed every
# SAVE_EVERY_CASES cases, at family boundaries, and on KeyboardInterrupt.
SAVE_EVERY_CASES = 10

# Five PL segments per braid crossing are enough to represent the crossing robustly
# while reducing projection cost substantially versus the previous value 9.
BRAID_SAMPLES_PER_CROSSING = 5
RETURN_SAMPLES = 31
SIDE_CONNECTION_SAMPLES = 9

# Safe canonical return corridors.
# The old long Bezier closures re-entered the side-pair band at large m.
SAFE_RETURN_Y = 1.85
SAFE_RETURN_RIGHT_MARGIN = 0.70
SAFE_RETURN_LEFT_X = -0.55

# Cheap geometry regression before the long Yamada sweep.
RUN_CANONICAL_PROJECTION_PREFLIGHT = True
CANONICAL_PREFLIGHT_M = [1, 5, 10, 25, 50, 100, 101, 200]

COMPUTE_CONFIG = {
    "schema": RESULT_SCHEMA_VERSION,
    "constructor_version": CONSTRUCTOR_VERSION,
    "rotation_order": ROTATION_ORDER,
    "use_canonical_zero_projection": USE_CANONICAL_ZERO_PROJECTION,
    "compute_frontier_diagnostics": COMPUTE_FRONTIER_DIAGNOSTICS,
    "normalize": NORMALIZE_YAMADA,
    "braid_samples_per_crossing": BRAID_SAMPLES_PER_CROSSING,
    "return_samples": RETURN_SAMPLES,
    "side_connection_samples": SIDE_CONNECTION_SAMPLES,
    "safe_return_y": SAFE_RETURN_Y,
    "safe_return_right_margin": SAFE_RETURN_RIGHT_MARGIN,
    "safe_return_left_x": SAFE_RETURN_LEFT_X,
}
COMPUTE_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(COMPUTE_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

RUN_CONFIG = {
    **COMPUTE_CONFIG,
    "dv_discovery_n": DV_DISCOVERY_N,
    "dv_heldout_n": DV_HELDOUT_N,
    "main_discovery_m": MAIN_DISCOVERY_M,
    "main_heldout_m": MAIN_HELDOUT_M,
    "compute_heldout_now": COMPUTE_HELDOUT_NOW,
    "save_every_cases": SAVE_EVERY_CASES,
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

print("Compute config :", COMPUTE_CONFIG_SHA256)
print("Request config :", RUN_CONFIG_SHA256)
print(
    "Main-family discovery m-range:",
    MAIN_DISCOVERY_M[0], "..", MAIN_DISCOVERY_M[-1],
    f"({len(MAIN_DISCOVERY_M)} values per family)",
)
print("Future held-out m-values:", MAIN_HELDOUT_M)
print("Canonical zero projection:", USE_CANONICAL_ZERO_PROJECTION)
print("Frontier diagnostics:", COMPUTE_FRONTIER_DIAGNOSTICS)
print("Checkpoint every:", SAVE_EVERY_CASES, "new cases")
print("FULL DATASET :", FULL_CSV)
print("SHARE DATASET:", SHARE_CSV)
print("HELD-OUT DATASET:", HELDOUT_CSV)

## Upstream Yamada sanity gate

In [ ]:
if RUN_UPSTREAM_SANITY:
    sanity_script = ROOT / "dev" / "run_yamada_sanity_checks.py"
    if not sanity_script.is_file():
        raise RuntimeError(f"Missing sanity script: {sanity_script}")

    proc = subprocess.run(
        [os.sys.executable, str(sanity_script)],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )

    print(proc.stdout)

    if proc.returncode:
        raise RuntimeError(
            "Upstream Yamada sanity checks failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )

    marker = "PASS: all published/independent Yamada sanity checks succeeded."
    if marker not in proc.stdout:
        raise RuntimeError("Expected upstream Yamada sanity success marker was not emitted.")

print("PASS: upstream Yamada sanity gate.")

## Geometry helpers

In [13]:
def join_paths(*paths: np.ndarray) -> np.ndarray:
    pieces = []
    for path in paths:
        path = np.asarray(path, dtype=float)
        if len(path) == 0:
            continue
        if pieces and np.allclose(pieces[-1][-1], path[0]):
            path = path[1:]
        if len(path):
            pieces.append(path)

    if not pieces:
        return np.empty((0, 3), dtype=float)
    return np.vstack(pieces)


def line(a, b, samples: int = 2) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (1.0 - t) * a + t * b


def bezier_cubic(p0, p1, p2, p3, samples: int = 41) -> np.ndarray:
    p0, p1, p2, p3 = [np.asarray(p, dtype=float) for p in (p0, p1, p2, p3)]
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def segment_distance_3d(p1, q1, p2, q2) -> float:
    p1 = np.asarray(p1, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    u = q1 - p1
    v = q2 - p2
    w = p1 - p2

    a = float(u @ u)
    b = float(u @ v)
    c = float(v @ v)
    d = float(u @ w)
    e = float(v @ w)

    eps = 1e-14

    if a <= eps and c <= eps:
        return float(np.linalg.norm(p1 - p2))
    if a <= eps:
        t = float(np.clip(e / c, 0.0, 1.0))
        return float(np.linalg.norm(p1 - (p2 + t * v)))
    if c <= eps:
        s = float(np.clip(-d / a, 0.0, 1.0))
        return float(np.linalg.norm((p1 + s * u) - p2))

    D = a * c - b * b
    sN = sD = D
    tN = tD = D

    if D < eps:
        sN = 0.0
        sD = 1.0
        tN = e
        tD = c
    else:
        sN = b * e - c * d
        tN = a * e - b * d

        if sN < 0.0:
            sN = 0.0
            tN = e
            tD = c
        elif sN > sD:
            sN = sD
            tN = e + b
            tD = c

    if tN < 0.0:
        tN = 0.0
        if -d < 0.0:
            sN = 0.0
        elif -d > a:
            sN = sD
        else:
            sN = -d
            sD = a
    elif tN > tD:
        tN = tD
        if (-d + b) < 0.0:
            sN = 0.0
        elif (-d + b) > a:
            sN = sD
        else:
            sN = -d + b
            sD = a

    sc = 0.0 if abs(sN) < eps else sN / sD
    tc = 0.0 if abs(tN) < eps else tN / tD

    return float(np.linalg.norm(w + sc * u - tc * v))


def audit_piecewise_linear_embedding(graph: nx.MultiGraph, *, tolerance: float = CONTACT_TOL) -> None:
    graph = ensure_embedding(graph, copy=True, normalize=True)

    segments = []
    for u, v, key, data in graph.edges(keys=True, data=True):
        pts = np.asarray(data["pts"], dtype=float)
        lengths = np.linalg.norm(np.diff(pts, axis=0), axis=1)
        if np.any(lengths <= tolerance):
            raise AssertionError(f"Degenerate segment in edge {(u, v, key)!r}.")
        for segment_index in range(len(pts) - 1):
            segments.append((u, v, key, segment_index, pts[segment_index], pts[segment_index + 1]))

    for i, first in enumerate(segments):
        u1, v1, k1, s1, p1, q1 = first
        for second in segments[i + 1:]:
            u2, v2, k2, s2, p2, q2 = second

            if (u1, v1, k1) == (u2, v2, k2) and abs(s1 - s2) <= 1:
                continue

            if (
                np.any(np.maximum(p1, q1) + tolerance < np.minimum(p2, q2))
                or np.any(np.maximum(p2, q2) + tolerance < np.minimum(p1, q1))
            ):
                continue

            allowed = False
            for node in {u1, v1}.intersection({u2, v2}):
                pos = np.asarray(graph.nodes[node]["pos"], dtype=float)
                first_touches = min(np.linalg.norm(p1 - pos), np.linalg.norm(q1 - pos)) <= tolerance
                second_touches = min(np.linalg.norm(p2 - pos), np.linalg.norm(q2 - pos)) <= tolerance
                if first_touches and second_touches:
                    allowed = True
                    break

            if allowed:
                continue

            if segment_distance_3d(p1, q1, p2, q2) <= tolerance:
                raise AssertionError(
                    "Unintended 3-D contact between "
                    f"{(u1, v1, k1, s1)!r} and {(u2, v2, k2, s2)!r}."
                )

## Shared positive-braid theta geometry and side-connection geometry

The braid geometry is identical for all three target families.

At the \(2m\) subdivision positions between consecutive crossings, the two braid strands are separated into a unique upper and lower vertex.  For the paired families, vertices are connected only in **non-overlapping consecutive pairs**.  The connecting arc is routed outside the braid strip (downward for lower pairs, upward for upper pairs), so the canonical projection introduces no additional crossings.

In [14]:
def positive_two_braid_geometry(
    q: int,
    *,
    samples_per_crossing: int = BRAID_SAMPLES_PER_CROSSING,
):
    q = int(q)
    if q <= 0 or q % 2 == 0:
        raise ValueError("q must be a positive odd integer.")
    samples_per_crossing = int(samples_per_crossing)
    if samples_per_crossing < 5 or samples_per_crossing % 2 == 0:
        raise ValueError("BRAID_SAMPLES_PER_CROSSING must be an odd integer >=5.")

    y0 = 0.62
    depth = 0.24

    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])

    x_left = 0.90
    x_right = x_left + float(q)

    count = q * samples_per_crossing + 1
    t = np.linspace(0.0, 1.0, count)
    knots = np.linspace(0.0, 1.0, q + 1)
    lane_values = y0 * ((-1.0) ** np.arange(q + 1))

    y_a = np.interp(t, knots, lane_values)
    z_a = depth * np.sin(np.pi * q * t)

    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])

    if not np.allclose(braid_a[-1], bottom_right):
        raise AssertionError("Odd positive braid A did not end on bottom lane.")
    if not np.allclose(braid_b[-1], top_right):
        raise AssertionError("Odd positive braid B did not end on top lane.")

    # Safe outside return corridors.
    #
    # The previous long Bezier returns were visually compact but, as q grew,
    # their xy projection entered the same band as the local side-pair arcs.
    # This produced accidental crossings.  These PL returns immediately leave
    # the repeated motif to x>x_right, travel in a far upper/lower corridor,
    # return at x<0, and then connect to U/V.  Hence they are disjoint in the
    # canonical xy projection from every local side-pair edge for all m.
    x_far_right = x_right + SAFE_RETURN_RIGHT_MARGIN
    x_far_left = SAFE_RETURN_LEFT_X

    top_return = np.array(
        [
            top_right,
            [x_far_right, y0, 0.0],
            [x_far_right, SAFE_RETURN_Y, 0.0],
            [x_far_left, SAFE_RETURN_Y, 0.0],
            [x_far_left, y0, 0.0],
            U,
        ],
        dtype=float,
    )

    bottom_return = np.array(
        [
            bottom_right,
            [x_far_right, -y0, 0.0],
            [x_far_right, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -y0, 0.0],
            V,
        ],
        dtype=float,
    )

    left_u = line(U, braid_a[0], 9)
    left_v = line(V, braid_b[0], 9)

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, RETURN_SAMPLES)
    exterior = np.column_stack(
        [
            -0.78 * np.cos(phi - np.pi),
            y0 * np.sin(phi),
            np.full_like(phi, 0.38),
        ]
    )
    exterior[0] = U
    exterior[-1] = V

    return {
        "U": U,
        "V": V,
        "braid_a": braid_a,
        "braid_b": braid_b,
        "left_u": left_u,
        "left_v": left_v,
        "top_return": top_return,
        "bottom_return": bottom_return,
        "exterior": exterior,
        "samples_per_crossing": samples_per_crossing,
    }



def lower_upper_nodes_for_boundary(graph: nx.MultiGraph, j: int):
    """Return the lower and upper subdivision node at braid boundary j."""
    a = ("a", int(j))
    b = ("b", int(j))

    ya = float(graph.nodes[a]["pos"][1])
    yb = float(graph.nodes[b]["pos"][1])

    if ya <= yb:
        return a, b
    return b, a


def side_connection_arc(
    p0: np.ndarray,
    p1: np.ndarray,
    *,
    direction: str,
    pair_index: int,
) -> np.ndarray:
    """
    Local outside-the-braid connection used by the degree<=3 paired families.

    Endpoints have the same y-coordinate in the canonical projection.  The arc
    bows away from the braid strip and receives a tiny alternating z-offset so
    distinct 3-D edges stay separated even before projection.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)

    if direction not in {"down", "up"}:
        raise ValueError("direction must be 'down' or 'up'.")

    sign = -1.0 if direction == "down" else 1.0
    span = float(p1[0] - p0[0])

    if span <= 0:
        raise ValueError("side_connection_arc expects p1 to lie to the right of p0.")

    bow = 0.48
    zoff = 0.06 * (1.0 if pair_index % 2 else -1.0)

    c1 = p0 + np.array([0.28 * span, sign * bow, zoff])
    c2 = p1 + np.array([-0.28 * span, sign * bow, -zoff])

    return bezier_cubic(
        p0,
        c1,
        c2,
        p1,
        samples=SIDE_CONNECTION_SAMPLES,
    )


def add_nonoverlapping_side_pairs(
    graph: nx.MultiGraph,
    *,
    add_bottom: bool,
    add_top: bool,
) -> None:
    """
    Pair boundaries (1,2), (3,4), ..., (2m-1,2m).

    Because q=2m+1, q-1=2m is even.  Every selected subdivision vertex therefore
    receives exactly ONE added edge, keeping its total degree <=3.
    """
    q = int(graph.graph["q"])

    if (q - 1) % 2:
        raise AssertionError("Expected q-1 to be even.")

    for j in range(1, q - 1, 2):
        jp = j + 1
        pair_index = (j + 1) // 2

        lower_j, upper_j = lower_upper_nodes_for_boundary(graph, j)
        lower_p, upper_p = lower_upper_nodes_for_boundary(graph, jp)

        if add_bottom:
            p0 = np.asarray(graph.nodes[lower_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[lower_p]["pos"], dtype=float)
            graph.add_edge(
                lower_j,
                lower_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="down",
                    pair_index=pair_index,
                ),
                role="bottom_pair_edge",
                pair_index=pair_index,
            )

        if add_top:
            p0 = np.asarray(graph.nodes[upper_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[upper_p]["pos"], dtype=float)
            graph.add_edge(
                upper_j,
                upper_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="up",
                    pair_index=pair_index,
                ),
                role="top_pair_edge",
                pair_index=pair_index,
            )


## Common subdivided theta skeleton

In [15]:
def subdivided_theta_skeleton(m: int) -> nx.MultiGraph:
    """
    Shared skeleton with vertices U, V and boundary vertices a_j, b_j, j=1..q-1.
    Constituent-cycle edges are subdivided at every boundary between consecutive crossings.
    """
    m = int(m)
    if m < 1:
        raise ValueError("m must be >=1.")

    q = 2 * m + 1
    geom = positive_two_braid_geometry(q)
    U = geom["U"]
    V = geom["V"]
    braid_a = geom["braid_a"]
    braid_b = geom["braid_b"]
    s = int(geom["samples_per_crossing"])

    graph = nx.MultiGraph()
    graph.add_node("U", pos=U.copy())
    graph.add_node("V", pos=V.copy())

    for j in range(1, q):
        index = j * s
        graph.add_node(("a", j), pos=braid_a[index].copy())
        graph.add_node(("b", j), pos=braid_b[index].copy())

    # Path A: U -> a1 -> ... -> a_(q-1) -> V
    first_a = join_paths(geom["left_u"], braid_a[: s + 1])
    graph.add_edge("U", ("a", 1), pts=first_a, role="constituent_cycle", path_family="A")

    for j in range(1, q - 1):
        pts = braid_a[j * s : (j + 1) * s + 1].copy()
        graph.add_edge(("a", j), ("a", j + 1), pts=pts, role="constituent_cycle", path_family="A")

    last_a = join_paths(braid_a[(q - 1) * s :], geom["bottom_return"])
    graph.add_edge(("a", q - 1), "V", pts=last_a, role="constituent_cycle", path_family="A")

    # Path B: U -> b_(q-1) -> ... -> b_1 -> V
    first_b = join_paths(geom["top_return"][::-1], braid_b[(q - 1) * s :][::-1])
    graph.add_edge("U", ("b", q - 1), pts=first_b, role="constituent_cycle", path_family="B")

    for j in range(q - 1, 1, -1):
        pts = braid_b[(j - 1) * s : j * s + 1][::-1].copy()
        graph.add_edge(("b", j), ("b", j - 1), pts=pts, role="constituent_cycle", path_family="B")

    last_b = join_paths(braid_b[: s + 1][::-1], geom["left_v"][::-1])
    graph.add_edge(("b", 1), "V", pts=last_b, role="constituent_cycle", path_family="B")

    graph.add_edge("U", "V", pts=geom["exterior"].copy(), role="exterior_theta_edge")

    graph.graph.update(
        parameter_name="m",
        m=m,
        q=q,
        constituent_knot=f"T(2,{q})",
        crossing_lower_bound=q,
        expected_canonical_crossings=q,
        expected_vertices=2 * q,
    )

    return ensure_embedding(graph, copy=False, normalize=True)

## Three target family constructors

In [16]:
def crosslinked_theta_ladder(m: int) -> nx.MultiGraph:
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    for j in range(1, q):
        a = ("a", j)
        b = ("b", j)

        p = np.asarray(graph.nodes[a]["pos"], dtype=float)
        p_other = np.asarray(graph.nodes[b]["pos"], dtype=float)

        graph.add_edge(
            a,
            b,
            pts=line(p, p_other, 2),
            role="rung",
            rung_index=j,
        )

    graph.graph.update(
        family="crosslinked_theta_ladder",
        family_label="Cross-linked theta ladder Lambda_m",
        expected_edges=3 * q,
        expected_rungs=q - 1,
        expected_bottom_pair_edges=0,
        expected_top_pair_edges=0,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def bottom_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^downarrow:
    pair consecutive LOWER boundary vertices non-overlappingly:
    (1,2), (3,4), ..., (2m-1,2m).
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=False,
    )

    # Base subdivided theta has 2q+1 edges; we add m=(q-1)/2 pair edges.
    pair_count = (q - 1) // 2

    graph.graph.update(
        family="bottom_paired_theta",
        family_label="Bottom-paired theta family P_down_m",
        expected_edges=2 * q + 1 + pair_count,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=0,
        # 2m lower boundary vertices + U,V are degree 3;
        # 2m upper boundary vertices remain degree 2.
        expected_degree_histogram={
            2: q - 1,
            3: q + 1,
        },
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def two_sided_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^{updownarrow}:
    add the same non-overlapping consecutive pairing on BOTH lower and upper sides.
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=True,
    )

    pair_count = (q - 1) // 2

    graph.graph.update(
        family="two_sided_paired_theta",
        family_label="Two-sided paired theta family P_updown_m",
        expected_edges=3 * q,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=pair_count,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)

## Known positive control: canonical Dobrynin–Vesnin theta family

In [17]:
def reference_dv_theta_graph(n: int, samples_per_crossing: int = 9) -> nx.MultiGraph:
    n = int(n)
    if n < 0:
        raise ValueError("n must be >=0.")

    y0 = 0.56
    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])

    x_left = 0.62
    x_right = 2.05 + 0.72 * max(n, 1)
    depth = 0.24

    if n == 0:
        t = np.linspace(0.0, 1.0, 24)
        y_a = np.full_like(t, y0)
        z_a = np.zeros_like(t)
    else:
        count = n * int(samples_per_crossing) + 1
        t = np.linspace(0.0, 1.0, count)
        knots = np.linspace(0.0, 1.0, n + 1)
        values = y0 * ((-1.0) ** np.arange(n + 1))
        y_a = np.interp(t, knots, values)
        z_a = depth * np.sin(np.pi * n * t)

    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])
    outer_y = 1.16

    top_return = bezier_cubic(
        top_right,
        [x_right + 0.46, outer_y, 0.0],
        [0.15, outer_y, 0.0],
        U,
        samples=max(70, 8 * max(n, 1)),
    )

    bottom_return = bezier_cubic(
        bottom_right,
        [x_right + 0.46, -outer_y, 0.0],
        [0.15, -outer_y, 0.0],
        V,
        samples=max(70, 8 * max(n, 1)),
    )

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, 70)
    exterior = np.column_stack(
        [
            -0.67 * np.cos(phi - np.pi),
            y0 * np.sin(phi),
            np.zeros_like(phi),
        ]
    )
    exterior[0] = U
    exterior[-1] = V

    graph = nx.MultiGraph()
    graph.add_node("u", pos=U.copy())
    graph.add_node("v", pos=V.copy())

    left_u = line(U, braid_a[0], 14)
    left_v = line(V, braid_b[0], 14)

    if n % 2:
        graph.add_edge("u", "v", pts=join_paths(left_u, braid_a, bottom_return), role="torus_a")
        graph.add_edge("u", "v", pts=join_paths(top_return[::-1], braid_b[::-1], left_v[::-1]), role="torus_b")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        graph.graph["abstract_type"] = "theta"
    else:
        graph.add_edge("u", "u", pts=join_paths(left_u, braid_a, top_return), role="torus_component_u")
        graph.add_edge("v", "v", pts=join_paths(left_v, braid_b, bottom_return), role="torus_component_v")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        graph.graph["abstract_type"] = "handcuff"

    graph.graph["expected_canonical_crossings"] = n
    return ensure_embedding(graph, copy=False, normalize=True)

## Hard certification

Every target family is certified to satisfy \(\Delta\le3\), to contain the same \(T(2,2m+1)\) constituent cycle, and to have the expected graph counts.

The expensive all-segment 3-D contact audit is **not** repeated for every \(m\).  Because the geometry is a translated repeated cell, it is run on representative values \(m=1,2,5,10\).  Each actual dataset row still receives the cheap exact combinatorial certification and the canonical projection is required to contain exactly \(q=2m+1\) crossings.

In [ ]:
def certify_connected(graph: nx.MultiGraph) -> None:
    if not nx.is_connected(nx.Graph(graph)):
        raise AssertionError("Graph is disconnected.")


def extract_constituent_cycle(graph: nx.MultiGraph) -> nx.MultiGraph:
    cycle = nx.MultiGraph()

    for node, data in graph.nodes(data=True):
        cycle.add_node(
            node,
            pos=np.asarray(data["pos"], dtype=float).copy(),
        )

    used_nodes = set()

    for u, v, key, data in graph.edges(keys=True, data=True):
        if data.get("role") != "constituent_cycle":
            continue

        cycle.add_edge(
            u,
            v,
            pts=np.asarray(data["pts"], dtype=float).copy(),
            role="constituent_cycle",
        )
        used_nodes.add(u)
        used_nodes.add(v)

    for node in list(cycle.nodes()):
        if node not in used_nodes:
            cycle.remove_node(node)

    return ensure_embedding(cycle, copy=False, normalize=True)


def certify_theta_derived_family(
    graph: nx.MultiGraph,
    *,
    require_cubic: bool,
    audit_3d_contacts: bool = False,
) -> None:
    certify_connected(graph)

    m = int(graph.graph["m"])
    q = int(graph.graph["q"])

    if q != 2 * m + 1:
        raise AssertionError("q != 2m+1.")

    if graph.number_of_nodes() != int(graph.graph["expected_vertices"]):
        raise AssertionError(
            f"Expected V={graph.graph['expected_vertices']}, "
            f"found {graph.number_of_nodes()}."
        )

    if graph.number_of_edges() != int(graph.graph["expected_edges"]):
        raise AssertionError(
            f"Expected E={graph.graph['expected_edges']}, "
            f"found {graph.number_of_edges()}."
        )

    role_counts = Counter(
        str(data.get("role", ""))
        for *_edge, data in graph.edges(data=True)
    )

    expected_role_counts = {
        "rung": int(graph.graph["expected_rungs"]),
        "bottom_pair_edge": int(graph.graph["expected_bottom_pair_edges"]),
        "top_pair_edge": int(graph.graph["expected_top_pair_edges"]),
    }

    for role, expected in expected_role_counts.items():
        actual = int(role_counts.get(role, 0))
        if actual != expected:
            raise AssertionError(
                f"Expected {expected} {role!r} edges, found {actual}."
            )

    degrees = [int(deg) for _, deg in graph.degree()]
    degree_histogram = dict(sorted(Counter(degrees).items()))

    if max(degrees) > 3:
        raise AssertionError(
            f"Subcubic requirement violated: Delta(G)={max(degrees)}."
        )

    expected_histogram = {
        int(k): int(v)
        for k, v in graph.graph["expected_degree_histogram"].items()
    }

    if degree_histogram != expected_histogram:
        raise AssertionError(
            "Degree histogram mismatch: "
            f"expected {expected_histogram}, found {degree_histogram}."
        )

    if require_cubic and any(deg != 3 for deg in degrees):
        raise AssertionError(
            f"Expected exactly cubic graph; histogram={degree_histogram}."
        )

    constituent = extract_constituent_cycle(graph)

    if not nx.is_connected(nx.Graph(constituent)):
        raise AssertionError("Constituent cycle is disconnected.")

    constituent_degrees = [int(deg) for _, deg in constituent.degree()]
    if any(deg != 2 for deg in constituent_degrees):
        raise AssertionError(
            "Constituent A∪B is not 2-regular; "
            f"histogram={dict(Counter(constituent_degrees))}."
        )

    # This zero-rotation constituent check is only used in representative preflight.
    if audit_3d_contacts:
        processor = PDCode(constituent)
        processor.compute(
            rotation_angles=(0.0, 0.0, 0.0),
            rotation_order=ROTATION_ORDER,
        )

        if len(processor.crossings) != q:
            raise AssertionError(
                f"Expected constituent to have q={q} canonical crossings, "
                f"detected {len(processor.crossings)}."
            )

        audit_piecewise_linear_embedding(graph)


# Representative expensive geometry audit only.
if RUN_REPRESENTATIVE_3D_CONTACT_AUDIT:
    for m in REPRESENTATIVE_AUDIT_M:
        for builder, cubic in [
            (crosslinked_theta_ladder, True),
            (bottom_paired_theta, False),
            (two_sided_paired_theta, True),
        ]:
            g = builder(m)
            certify_theta_derived_family(
                g,
                require_cubic=cubic,
                audit_3d_contacts=True,
            )

    print(
        "PASS: representative 3-D contact/constituent audits completed for m =",
        REPRESENTATIVE_AUDIT_M,
    )

# Cheap structural preflight at a few larger values.
for m in sorted(set([1, 2, 5, 10, min(25, MAX_DISCOVERY_M), MAX_DISCOVERY_M])):
    if m < 1:
        continue

    for builder, cubic in [
        (crosslinked_theta_ladder, True),
        (bottom_paired_theta, False),
        (two_sided_paired_theta, True),
    ]:
        g = builder(m)
        certify_theta_derived_family(
            g,
            require_cubic=cubic,
            audit_3d_contacts=False,
        )

for n in range(3, 21):
    dv = reference_dv_theta_graph(n)
    certify_connected(dv)

print("PASS: all three target constructors are certified Delta(G)<=3.")

## Canonical-projection regression at large \(m\)

Before the long invariant sweep, project representative large family members
(including the held-out-test scale) and require the **full graph** to contain
exactly \(q=2m+1\) crossings.  This catches embedding-constructor errors before
Yamada evaluation.

In [ ]:
if RUN_CANONICAL_PROJECTION_PREFLIGHT:
    builders = [
        ("crosslinked_theta_ladder", crosslinked_theta_ladder),
        ("bottom_paired_theta", bottom_paired_theta),
        ("two_sided_paired_theta", two_sided_paired_theta),
    ]

    for m in CANONICAL_PREFLIGHT_M:
        q = 2 * int(m) + 1

        for family_name, builder in builders:
            graph = builder(int(m))

            processor = PDCode(graph)
            processor.compute(
                rotation_angles=(0.0, 0.0, 0.0),
                rotation_order=ROTATION_ORDER,
            )

            actual = len(processor.crossings)

            if actual != q:
                raise AssertionError(
                    f"Canonical-projection preflight failed for "
                    f"{family_name}, m={m}: expected q={q}, got {actual}."
                )

            print(
                f"PASS canonical projection: "
                f"{family_name:30s} m={m:3d} crossings={actual:3d}"
            )

    print(
        "PASS: safe-return geometry preserves exactly q=2m+1 canonical "
        "crossings through m=200."
    )

## Fast canonical projection

These families are defined by the displayed canonical braid diagrams.  Therefore the dataset does **not** search over random rotations.  It computes the zero-rotation PD code once, asserts that the target families have exactly \(q=2m+1\) crossings, and evaluates Yamada.

This removes the largest avoidable wall-time component of the earlier implementation.  Optional frontier diagnostics can be re-enabled with `COMPUTE_FRONTIER_DIAGNOSTICS=True`.

In [20]:
def factorized_frontier_width(processor) -> dict:
    """Optional diagnostic only; disabled by default for maximum throughput."""
    yamada = Yamada(
        vertices=list(processor.vertices.values()),
        crossings=list(processor.crossings.values()),
        arcs=list(processor.arcs.values()),
    )

    prepared = yamada._prepare_compact_state_builder()
    data = build_factorized_frontier(prepared)

    factor_count = len(data["factor_types"])
    ports_by_factor = [[] for _ in range(factor_count)]

    for port, factor in enumerate(data["port_factor"]):
        ports_by_factor[int(factor)].append(port)

    active = []
    processed = set()
    peak_live_ports = 0
    max_boundary_ports = 0

    for factor in data["factor_order"]:
        factor = int(factor)
        active.extend(ports_by_factor[factor])
        peak_live_ports = max(peak_live_ports, len(active))
        processed.add(factor)

        active = [
            port
            for port in active
            if int(
                data["port_factor"][
                    int(data["wire_partner"][port])
                ]
            )
            not in processed
        ]

        max_boundary_ports = max(
            max_boundary_ports,
            len(active),
        )

    if active:
        raise RuntimeError("Factorized frontier planner did not close.")

    return {
        "crossings_after_RII": len(prepared.crossing_ids),
        "peak_live_ports": int(peak_live_ports),
        "max_boundary_ports": int(max_boundary_ports),
        "factor_count": int(factor_count),
    }


def embedding_hash(graph: nx.MultiGraph) -> str:
    payload = []

    for node, data in sorted(
        graph.nodes(data=True),
        key=lambda item: repr(item[0]),
    ):
        payload.append(
            (
                "node",
                repr(node),
                np.asarray(data["pos"], dtype=float).round(12).tolist(),
            )
        )

    edge_payload = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        edge_payload.append(
            (
                repr(u),
                repr(v),
                int(key),
                str(data.get("role", "")),
                np.asarray(data["pts"], dtype=float).round(12).tolist(),
            )
        )

    payload.extend(
        ("edge", *entry)
        for entry in sorted(
            edge_payload,
            key=lambda item: (item[0], item[1], item[2], item[3]),
        )
    )

    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()


def projection_cache_key(
    *,
    family: str,
    parameter_value: int,
    graph_hash: str,
) -> tuple[str, Path]:
    payload = {
        "schema": 2,
        "constructor_version": CONSTRUCTOR_VERSION,
        "git_commit": GIT_COMMIT,
        "family": family,
        "parameter_value": int(parameter_value),
        "graph_hash": graph_hash,
        "rotation_order": ROTATION_ORDER,
        "canonical_zero": True,
        "frontier_diagnostics": COMPUTE_FRONTIER_DIAGNOSTICS,
    }

    key = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()

    safe_family = family.replace("/", "_")

    path = (
        PROJECTION_CACHE_DIR
        / f"{safe_family}__p{parameter_value}__{key[:20]}.pkl"
    )

    return key, path


def _processor_payload(processor) -> dict:
    return {
        "vertices": dict(processor.vertices),
        "crossings": dict(processor.crossings),
        "arcs": dict(processor.arcs),
    }


def _processor_from_payload(payload: dict):
    return SimpleNamespace(
        vertices=dict(payload["vertices"]),
        crossings=dict(payload["crossings"]),
        arcs=dict(payload["arcs"]),
    )


def save_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
    projection,
    frontier: dict,
) -> None:
    if not WRITE_PROJECTION_CACHE:
        return

    payload = {
        "schema": 2,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        **_processor_payload(projection.processor),
    }

    tmp = path.with_suffix(path.suffix + ".tmp")

    with tmp.open("wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


def load_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
):
    if not REUSE_PROJECTION_CACHE or not path.exists():
        return None

    try:
        with path.open("rb") as handle:
            payload = pickle.load(handle)

        if payload.get("schema") != 2:
            return None
        if payload.get("cache_key") != cache_key:
            return None
        if payload.get("graph_hash") != graph_hash:
            return None

        processor = _processor_from_payload(payload)

        return {
            "processor": processor,
            "rotation_angles": payload["rotation_angles"],
            "rotation_order": payload["rotation_order"],
            "pd_code": payload["pd_code"],
            "num_crossings": int(payload["num_crossings"]),
            "frontier": dict(payload.get("frontier", {})),
            "source": "cache",
            "candidate_records": [],
        }

    except Exception as exc:
        print(
            f"Ignoring stale/broken projection cache {path.name}: "
            f"{type(exc).__name__}: {exc}"
        )
        return None


def choose_projection(
    graph: nx.MultiGraph,
    *,
    family: str,
    parameter_value: int,
):
    """
    Fast path: one canonical zero-rotation projection only.

    Projection independence is already covered by the repository sanity suite;
    here the canonical projection is part of the family definition.
    """
    graph_hash = embedding_hash(graph)

    cache_key, cache_path = projection_cache_key(
        family=family,
        parameter_value=parameter_value,
        graph_hash=graph_hash,
    )

    cached = load_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
    )

    if cached is not None:
        cached["cache_key"] = cache_key
        cached["graph_hash"] = graph_hash
        return cached

    start = time.perf_counter()

    projection = select_projection(
        graph,
        rotation_angles=(0.0, 0.0, 0.0),
        rotation_order=ROTATION_ORDER,
        num_rotation_samples=1,
    )

    if COMPUTE_FRONTIER_DIAGNOSTICS:
        frontier = factorized_frontier_width(projection.processor)
    else:
        frontier = {}

    projection_seconds = time.perf_counter() - start

    save_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
        projection=projection,
        frontier=frontier,
    )

    return {
        "processor": projection.processor,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        "candidate_records": [
            {
                "angles": [0.0, 0.0, 0.0],
                "crossings": int(projection.num_crossings),
                "canonical": True,
            }
        ],
        "source": "computed",
        "projection_seconds": projection_seconds,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
    }

## Exact Laurent representation

In [21]:
def laurent_coefficients_exact(
    expr: sp.Expr,
    variable: sp.Symbol,
) -> dict[int, int]:
    """
    Fast exact Laurent extraction.

    The Yamada result is already an expanded Laurent polynomial.  Avoiding
    sp.cancel() on the whole expression substantially reduces post-processing
    time for large m.
    """
    expr = sp.expand(expr)

    if expr == 0:
        return {}

    coefficients: dict[int, sp.Expr] = {}

    for term in sp.Add.make_args(expr):
        coefficient, exponent = term.as_coeff_exponent(variable)

        if exponent.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent exponent in term {term!r}."
            )

        if variable in coefficient.free_symbols:
            raise ValueError(
                f"Could not isolate Laurent coefficient in term {term!r}."
            )

        exponent = int(exponent)
        coefficients[exponent] = (
            coefficients.get(exponent, sp.Integer(0))
            + coefficient
        )

    result: dict[int, int] = {}

    for exponent, coefficient in sorted(coefficients.items()):
        coefficient = sp.expand(coefficient)

        if coefficient == 0:
            continue

        if coefficient.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent coefficient: {coefficient!r}."
            )

        result[int(exponent)] = int(coefficient)

    return result


def exact_same(left: sp.Expr, right: sp.Expr) -> bool:
    return sp.expand(left - right) == 0

## Family registry

In [22]:
@dataclass(frozen=True)
class FamilySpec:
    key: str
    label: str
    parameter_name: str
    builder: Callable[[int], nx.MultiGraph]
    family_definition: str
    spatial_definition: str
    literature_role: str
    require_cubic: bool
    has_crossing_lower_bound: bool
    discovery_values: tuple[int, ...]
    heldout_values: tuple[int, ...]


FAMILIES = [
    FamilySpec(
        key="dv_theta",
        label="Dobrynin–Vesnin Theta(n)",
        parameter_name="n",
        builder=reference_dv_theta_graph,
        family_definition=(
            "Published Dobrynin–Vesnin two-strand Theta(n) spatial family."
        ),
        spatial_definition=(
            "Canonical n-crossing two-strand diagram plus exterior theta edge."
        ),
        literature_role="known_formula_positive_control",
        require_cubic=False,
        has_crossing_lower_bound=False,
        discovery_values=tuple(DV_DISCOVERY_N),
        heldout_values=tuple(DV_HELDOUT_N),
    ),
    FamilySpec(
        key="crosslinked_theta_ladder",
        label="Cross-linked theta ladder Lambda_m",
        parameter_name="m",
        builder=crosslinked_theta_ladder,
        family_definition=(
            "At every braid boundary, connect the upper and lower subdivision "
            "vertices by one rung."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus 2m rungs."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=True,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
    FamilySpec(
        key="bottom_paired_theta",
        label="Bottom-paired theta family P_down_m",
        parameter_name="m",
        builder=bottom_paired_theta,
        family_definition=(
            "Pair consecutive lower braid-boundary vertices non-overlappingly: "
            "(1,2),(3,4),...,(2m-1,2m)."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus m local "
            "bottom-side pairing arcs; maximum degree is 3."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=False,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
    FamilySpec(
        key="two_sided_paired_theta",
        label="Two-sided paired theta family P_updown_m",
        parameter_name="m",
        builder=two_sided_paired_theta,
        family_definition=(
            "Apply the same non-overlapping consecutive pairing on both the "
            "lower and upper braid-boundary vertices."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus m lower and "
            "m upper local pairing arcs; the graph is cubic."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=True,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
]

SPEC_BY_KEY = {
    spec.key: spec
    for spec in FAMILIES
}


def requested_values(spec: FamilySpec) -> list[int]:
    values = set(spec.discovery_values)

    if COMPUTE_HELDOUT_NOW:
        values |= set(spec.heldout_values)

    return sorted(values)


def split_for(spec: FamilySpec, value: int) -> str:
    if value in spec.discovery_values:
        return "discovery"

    if value in spec.heldout_values:
        return "heldout_test"

    raise ValueError(
        f"{spec.key}, {spec.parameter_name}={value} belongs to no split."
    )

## Independent exact Yamada evaluation

Each row is independently constructed, projected, and evaluated by KnottedGraph's compiled production Yamada backend.  No earlier \(m\), recurrence, transfer matrix, or candidate formula is used.

The timing columns separate construction, certification, projection, optimized Yamada contraction, Laurent post-processing, and total wall time so any remaining bottleneck is visible.

In [23]:
CSV_FIELDS = [
    "case_id",
    "split",
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "branch",
    "git_commit",
    "constructor_version",
    "run_config_sha256",
    "compute_config_sha256",
    "embedding_sha256",
    "projection_cache_key",
    "projection_source",
    "rotation_angles_json",
    "rotation_order",
    "candidate_projection_scores_json",
    "vertices",
    "edges",
    "components",
    "beta1",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "degree_histogram_json",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "canonical_crossing_count_pass",
    "crossing_lower_bound_pass",
    "crossings_after_RII",
    "factorized_peak_live_ports",
    "factorized_max_boundary_ports",
    "factorized_factor_count",
    "pd_code",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
    "construction_seconds",
    "certification_seconds",
    "projection_seconds",
    "yamada_seconds",
    "postprocess_seconds",
    "case_seconds",
    "wall_minus_yamada_seconds",
    "status",
    "error_type",
    "error_message",
]


def graph_summary(graph: nx.MultiGraph) -> dict:
    degrees = [int(deg) for _, deg in graph.degree()]
    components = nx.number_connected_components(nx.Graph(graph))
    beta1 = (
        graph.number_of_edges()
        - graph.number_of_nodes()
        + components
    )

    return {
        "vertices": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "components": components,
        "beta1": beta1,
        "max_degree": max(degrees),
        "is_subcubic": max(degrees) <= 3,
        "is_cubic": all(deg == 3 for deg in degrees),
        "degree_histogram_json": json.dumps(
            dict(sorted(Counter(degrees).items())),
            separators=(",", ":"),
        ),
    }


def _frontier_field(frontier: dict, key: str):
    value = frontier.get(key, "")
    return value if value != "" else ""


def evaluate_family_case(
    spec: FamilySpec,
    parameter_value: int,
) -> dict:
    total_start = time.perf_counter()
    value = int(parameter_value)

    # ---------------- construction ----------------
    stage = time.perf_counter()
    graph = spec.builder(value)
    construction_seconds = time.perf_counter() - stage

    # ---------------- cheap structural certification ----------------
    stage = time.perf_counter()
    certify_connected(graph)

    if spec.key in {
        "crosslinked_theta_ladder",
        "bottom_paired_theta",
        "two_sided_paired_theta",
    }:
        certify_theta_derived_family(
            graph,
            require_cubic=spec.require_cubic,
            audit_3d_contacts=False,
        )

    summary = graph_summary(graph)
    certification_seconds = time.perf_counter() - stage

    # ---------------- canonical projection ----------------
    stage = time.perf_counter()
    chosen = choose_projection(
        graph,
        family=spec.key,
        parameter_value=value,
    )
    projection_seconds = time.perf_counter() - stage

    canonical_crossing_count_pass = ""

    if spec.key in {
        "crosslinked_theta_ladder",
        "bottom_paired_theta",
        "two_sided_paired_theta",
    }:
        expected_crossings = int(graph.graph["expected_canonical_crossings"])
        canonical_crossing_count_pass = (
            int(chosen["num_crossings"])
            == expected_crossings
        )

        if not canonical_crossing_count_pass:
            raise AssertionError(
                f"{spec.key}, m={value}: canonical projection has "
                f"{chosen['num_crossings']} crossings; expected exactly "
                f"{expected_crossings}. This indicates an accidental extra "
                "crossing or a lost braid crossing."
            )

    lower_bound = (
        int(graph.graph["crossing_lower_bound"])
        if spec.has_crossing_lower_bound
        else ""
    )

    lower_bound_pass = ""

    if spec.has_crossing_lower_bound:
        lower_bound_pass = (
            int(chosen["num_crossings"])
            >= int(lower_bound)
        )

        if not lower_bound_pass:
            raise AssertionError(
                f"{spec.key}, {spec.parameter_name}={value}: selected projection "
                f"has {chosen['num_crossings']} crossings, below constituent-knot "
                f"lower bound {lower_bound}."
            )

    # ---------------- optimized exact Yamada ----------------
    stage = time.perf_counter()

    computer = Yamada(
        vertices=list(chosen["processor"].vertices.values()),
        crossings=list(chosen["processor"].crossings.values()),
        arcs=list(chosen["processor"].arcs.values()),
    )

    polynomial = sp.expand(
        computer.compute(
            A,
            normalize=NORMALIZE_YAMADA,
        )
    )

    yamada_seconds = time.perf_counter() - stage

    # ---------------- exact polynomial post-processing ----------------
    stage = time.perf_counter()

    coeffs = laurent_coefficients_exact(
        polynomial,
        A,
    )

    min_exp = min(coeffs) if coeffs else ""
    max_exp = max(coeffs) if coeffs else ""
    span = max_exp - min_exp if coeffs else 0

    polynomial_text = sp.sstr(polynomial)
    coefficients_text = json.dumps(
        coeffs,
        sort_keys=True,
        separators=(",", ":"),
    )

    postprocess_seconds = time.perf_counter() - stage
    case_seconds = time.perf_counter() - total_start

    frontier = chosen.get("frontier", {})
    q_value = (
        int(graph.graph.get("q", ""))
        if graph.graph.get("q", "") != ""
        else ""
    )
    m_value = value if spec.parameter_name == "m" else ""

    return {
        "case_id": f"{spec.key}__{spec.parameter_name}{value}",
        "split": split_for(spec, value),
        "family": spec.key,
        "family_label": spec.label,
        "parameter_name": spec.parameter_name,
        "parameter_value": value,
        "m": m_value,
        "q": q_value,
        "family_definition": spec.family_definition,
        "spatial_definition": spec.spatial_definition,
        "literature_role": spec.literature_role,
        "branch": CURRENT_BRANCH,
        "git_commit": GIT_COMMIT,
        "constructor_version": CONSTRUCTOR_VERSION,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "compute_config_sha256": COMPUTE_CONFIG_SHA256,
        "embedding_sha256": chosen["graph_hash"],
        "projection_cache_key": chosen["cache_key"],
        "projection_source": chosen["source"],
        "rotation_angles_json": json.dumps(
            (
                None
                if chosen["rotation_angles"] is None
                else [
                    float(x)
                    for x in chosen["rotation_angles"]
                ]
            ),
            separators=(",", ":"),
        ),
        "rotation_order": chosen["rotation_order"],
        "candidate_projection_scores_json": json.dumps(
            chosen.get("candidate_records", []),
            separators=(",", ":"),
        ),
        **summary,
        "constituent_knot": graph.graph.get("constituent_knot", ""),
        "crossing_lower_bound": lower_bound,
        "selected_crossings": int(chosen["num_crossings"]),
        "canonical_crossing_count_pass": canonical_crossing_count_pass,
        "crossing_lower_bound_pass": lower_bound_pass,
        "crossings_after_RII": _frontier_field(
            frontier,
            "crossings_after_RII",
        ),
        "factorized_peak_live_ports": _frontier_field(
            frontier,
            "peak_live_ports",
        ),
        "factorized_max_boundary_ports": _frontier_field(
            frontier,
            "max_boundary_ports",
        ),
        "factorized_factor_count": _frontier_field(
            frontier,
            "factor_count",
        ),
        "pd_code": chosen["pd_code"],
        "yamada_A_normalized": polynomial_text,
        "laurent_min_exponent": min_exp,
        "laurent_max_exponent": max_exp,
        "laurent_span": span,
        "laurent_term_count": len(coeffs),
        "laurent_coefficients_json": coefficients_text,
        "construction_seconds": construction_seconds,
        "certification_seconds": certification_seconds,
        "projection_seconds": projection_seconds,
        "yamada_seconds": yamada_seconds,
        "postprocess_seconds": postprocess_seconds,
        "case_seconds": case_seconds,
        "wall_minus_yamada_seconds": case_seconds - yamada_seconds,
        "status": "success",
        "error_type": "",
        "error_message": "",
    }


def read_csv(path: Path) -> list[dict]:
    if not path.exists():
        return []

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        return list(csv.DictReader(handle))


def atomic_write_rows(
    rows: list[dict],
    path: Path,
    *,
    fields: list[str],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            extrasaction="ignore",
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(
                {
                    field: row.get(field, "")
                    for field in fields
                }
            )

        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


family_rank = {
    spec.key: i
    for i, spec in enumerate(FAMILIES)
}


def sort_rows(rows: list[dict]) -> list[dict]:
    return sorted(
        rows,
        key=lambda row: (
            family_rank.get(
                row.get("family", ""),
                999,
            ),
            int(
                row.get(
                    "parameter_value",
                    0,
                )
            ),
        ),
    )

## Generate the discovery dataset

The full/share CSVs are checkpointed every `SAVE_EVERY_CASES` newly computed rows rather than after every case.  A manual `KeyboardInterrupt` always flushes all completed rows first.

In [ ]:
existing = (
    read_csv(FULL_CSV)
    if RESUME
    else []
)

rows_by_id = {
    row["case_id"]: row
    for row in existing
}

for case_id, row in list(rows_by_id.items()):
    if row.get("status") != "success":
        continue

    if row.get("branch") != CURRENT_BRANCH:
        raise RuntimeError(
            f"Existing row {case_id} is from another branch."
        )

    if row.get("git_commit") != GIT_COMMIT:
        raise RuntimeError(
            f"Existing row {case_id} is from commit "
            f"{row.get('git_commit')}; current commit is {GIT_COMMIT}. "
            "Rename/remove the old CSV."
        )

    if row.get("constructor_version") != CONSTRUCTOR_VERSION:
        raise RuntimeError(
            f"Existing row {case_id} uses another constructor version."
        )


SHARE_FIELDS = [
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
]


def refresh_share_csv() -> list[dict]:
    discovery_rows = []

    for row in rows_by_id.values():
        if row.get("status") != "success":
            continue

        spec = SPEC_BY_KEY.get(
            row.get("family", "")
        )

        if spec is None:
            continue

        value = int(
            row["parameter_value"]
        )

        if value not in spec.discovery_values:
            continue

        discovery_rows.append(row)

    atomic_write_rows(
        sort_rows(discovery_rows),
        SHARE_CSV,
        fields=SHARE_FIELDS,
    )

    return discovery_rows


def flush_ledgers() -> list[dict]:
    atomic_write_rows(
        sort_rows(
            list(rows_by_id.values())
        ),
        FULL_CSV,
        fields=CSV_FIELDS,
    )

    return refresh_share_csv()


# Initial checkpoint.
flush_ledgers()

failures = []
new_cases_since_flush = 0

for spec in FAMILIES:
    print(
        f"\n=== {spec.label} ===",
        flush=True,
    )

    for value in requested_values(spec):
        case_id = (
            f"{spec.key}__"
            f"{spec.parameter_name}{value}"
        )

        previous = rows_by_id.get(
            case_id
        )

        reusable = (
            RESUME
            and previous is not None
            and previous.get("status") == "success"
            and previous.get("branch") == CURRENT_BRANCH
            and previous.get("git_commit") == GIT_COMMIT
            and previous.get("constructor_version") == CONSTRUCTOR_VERSION
        )

        if reusable:
            print(
                f"SKIP {case_id:46s} "
                f"crossings={previous['selected_crossings']} "
                f"Yamada={float(previous['yamada_seconds']):.4g}s "
                f"wall={float(previous['case_seconds']):.4g}s",
                flush=True,
            )
            continue

        try:
            row = evaluate_family_case(
                spec,
                value,
            )

            rows_by_id[case_id] = row
            new_cases_since_flush += 1

            family_done_values = [
                int(r["parameter_value"])
                for r in rows_by_id.values()
                if (
                    r.get("status") == "success"
                    and r.get("family") == spec.key
                    and int(r["parameter_value"]) in spec.discovery_values
                )
            ]

            max_done = max(
                family_done_values,
                default=None,
            )

            print(
                f"PASS {case_id:46s} "
                f"V={row['vertices']:4d} "
                f"crossings={row['selected_crossings']:4d} "
                f"terms={row['laurent_term_count']:4d} "
                f"proj={row['projection_seconds']:.4g}s "
                f"Yamada={row['yamada_seconds']:.4g}s "
                f"post={row['postprocess_seconds']:.4g}s "
                f"wall={row['case_seconds']:.4g}s "
                f"| max discovery parameter={max_done}",
                flush=True,
            )

            if new_cases_since_flush >= SAVE_EVERY_CASES:
                flush_ledgers()
                new_cases_since_flush = 0
                print(
                    "  checkpoint saved",
                    flush=True,
                )

        except KeyboardInterrupt:
            flush_ledgers()

            print(
                "\nInterrupted by user. All completed exact rows and "
                "the current share CSV have been saved.",
                flush=True,
            )

            raise

        except Exception as exc:
            error_row = {
                "case_id": case_id,
                "split": split_for(
                    spec,
                    value,
                ),
                "family": spec.key,
                "family_label": spec.label,
                "parameter_name": spec.parameter_name,
                "parameter_value": value,
                "m": (
                    value
                    if spec.parameter_name == "m"
                    else ""
                ),
                "q": (
                    2 * value + 1
                    if spec.parameter_name == "m"
                    else ""
                ),
                "family_definition": spec.family_definition,
                "spatial_definition": spec.spatial_definition,
                "literature_role": spec.literature_role,
                "branch": CURRENT_BRANCH,
                "git_commit": GIT_COMMIT,
                "constructor_version": CONSTRUCTOR_VERSION,
                "run_config_sha256": RUN_CONFIG_SHA256,
                "compute_config_sha256": COMPUTE_CONFIG_SHA256,
                "status": "error",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }

            rows_by_id[case_id] = error_row
            failures.append(
                (
                    case_id,
                    type(exc).__name__,
                    str(exc),
                )
            )
            new_cases_since_flush += 1

            print(
                f"FAIL {case_id}: "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )

            if new_cases_since_flush >= SAVE_EVERY_CASES:
                flush_ledgers()
                new_cases_since_flush = 0

            if FAIL_FAST:
                flush_ledgers()
                raise

    # Always checkpoint at a family boundary.
    flush_ledgers()
    new_cases_since_flush = 0

print("\nFull dataset:", FULL_CSV.resolve())
print("Current share data:", SHARE_CSV.resolve())

if failures:
    print(
        f"\nWARNING: {len(failures)} case(s) failed, but every successful "
        "discovery row has been checkpointed."
    )

    for case_id, kind, message in failures:
        print(
            f"  {case_id}: {kind}: {message}"
        )

## Export blind discovery / held-out files

In [ ]:
full_rows = read_csv(
    FULL_CSV
)

SHARE_FIELDS = [
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
]

HELDOUT_FIELDS = SHARE_FIELDS + [
    "case_id",
    "branch",
    "git_commit",
    "embedding_sha256",
    "pd_code",
    "construction_seconds",
    "projection_seconds",
    "yamada_seconds",
    "postprocess_seconds",
    "case_seconds",
]

discovery_rows = []
heldout_rows = []

for row in full_rows:
    if row.get("status") != "success":
        continue

    spec = SPEC_BY_KEY.get(
        row.get("family", "")
    )

    if spec is None:
        continue

    value = int(
        row["parameter_value"]
    )

    if value in spec.discovery_values:
        discovery_rows.append(row)

    if (
        COMPUTE_HELDOUT_NOW
        and value in spec.heldout_values
    ):
        heldout_rows.append(row)

atomic_write_rows(
    sort_rows(discovery_rows),
    SHARE_CSV,
    fields=SHARE_FIELDS,
)

atomic_write_rows(
    sort_rows(heldout_rows),
    HELDOUT_CSV,
    fields=HELDOUT_FIELDS,
)

print(
    f"SHARE DATASET: {SHARE_CSV.resolve()} "
    f"({len(discovery_rows)} completed rows)"
)

print(
    f"HELD-OUT PRECOMPUTED: {len(heldout_rows)} "
    f"(COMPUTE_HELDOUT_NOW={COMPUTE_HELDOUT_NOW})"
)

for spec in FAMILIES:
    done = sorted(
        int(row["parameter_value"])
        for row in discovery_rows
        if row["family"] == spec.key
    )

    requested = list(
        spec.discovery_values
    )

    missing = [
        value
        for value in requested
        if value not in set(done)
    ]

    print(
        f"{spec.key:30s} "
        f"done={len(done):3d}/{len(requested):3d}; "
        f"max_done={max(done) if done else None}; "
        f"remaining={len(missing)}"
    )

## Export a formula-discovery dataset

After the discovery sweep has completed as far as practical, use the share CSV as the public fitting dataset. It contains the exact rows needed to search for Laurent-polynomial recurrences, factorizations, rational generating functions, parity sectors, or finite exponential-sum forms.

Freeze any machine-evaluable SymPy candidates before computing the held-out values. All three target families satisfy \(\Delta\le3\), share the constituent \(T(2,2m+1)\), and have canonical crossing count \(2m+1\).

# Held-out exact test after formulas are proposed

In [25]:
# --------------------------------------------------------------------
# PASTE FROZEN FORMULA CANDIDATES HERE ONLY AFTER THE SHARE_CSV HAS BEEN EXPORTED.
# --------------------------------------------------------------------
#
# Example syntax only:
#
# FORMULA_CANDIDATES = {
#     "dv_theta": lambda n, A: ...,
#     "crosslinked_theta_ladder": lambda m, A: ...,
#     "bottom_paired_theta": lambda m, A: ...,
#     "two_sided_paired_theta": lambda m, A: ...,
# }

FORMULA_CANDIDATES = {
    "crosslinked_theta_ladder": lambda m, A: (
        (A**2 + 1)**(2*m + 1)
        - A**(2*m)
          * (A**4 + A**2 + 1)
          * (A**2 - A + 1)**(2*m)
        - A**(8*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
    ),

    "bottom_paired_theta": lambda m, A: (
        (A**2 + 1)**(m + 1)
        - A**(2*m)
          * (A**4 + A**2 + 1)
          * (A**2 - A + 1)**m
        + (-1)**(m + 1)
          * A**(7*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
    ),

    "two_sided_paired_theta": lambda m, A: (
        (A**2 + 1)**(m + 1)
        * (A**2 - A + 1)**m

        + (-1)**(m + 1)
          * A**(2*m)
          * (A**4 + A**2 + 1)
          * (A - 1)**m
          * (2*A**2 - A + 2)**m

        - A**(5*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
          * (A + 1)**m
          * (A**2 - A + 1)**m
    ),
}

## Strongest held-out test: compute reserved values for the first time

The default discovery run leaves the held-out file empty.

After all candidate formulas are frozen, set

```python
RECOMPUTE_HELDOUT_FRESH = True
```

and run the final cell.  KnottedGraph will then compute \(m=101,125,150,200\) for the new families for the first time.

In [ ]:
def parse_candidate(candidate, parameter_value: int) -> sp.Expr:
    if callable(candidate):
        return sp.sympify(candidate(int(parameter_value), A))
    if isinstance(candidate, str):
        return sp.sympify(
            candidate,
            locals={
                "A": A,
                "m": sp.Integer(int(parameter_value)),
                "n": sp.Integer(int(parameter_value)),
                "sigma": SIGMA,
            },
        )
    return sp.sympify(candidate)


def parse_actual_polynomial(text: str) -> sp.Expr:
    return sp.sympify(text, locals={"A": A})


RECOMPUTE_HELDOUT_FRESH = True

if RECOMPUTE_HELDOUT_FRESH:
    if not FORMULA_CANDIDATES:
        raise RuntimeError("Paste frozen formula candidates first.")

    fresh_results = []

    for spec in FAMILIES:
        if spec.key not in FORMULA_CANDIDATES:
            continue

        for value in spec.heldout_values:
            print(f"FRESH HELD-OUT {spec.key} {spec.parameter_name}={value}", flush=True)

            fresh_row = evaluate_family_case(spec, value)
            actual = parse_actual_polynomial(fresh_row["yamada_A_normalized"])
            predicted = parse_candidate(FORMULA_CANDIDATES[spec.key], value)

            passed = exact_same(sp.expand(actual), sp.expand(predicted))
            fresh_results.append(
                {
                    "family": spec.key,
                    "parameter_name": spec.parameter_name,
                    "parameter_value": value,
                    "pass": passed,
                    "fresh_yamada_seconds": fresh_row["yamada_seconds"],
                }
            )

            print(
                f"{'PASS' if passed else 'FAIL'} "
                f"{spec.key} {spec.parameter_name}={value}; "
                f"fresh Yamada={fresh_row['yamada_seconds']:.4g}s",
                flush=True,
            )

    if not all(result["pass"] for result in fresh_results):
        raise AssertionError("At least one frozen candidate failed fresh exact held-out recomputation.")

    print("\nPASS: every frozen candidate survived fresh exact KnottedGraph held-out extrapolation.")

## Paper interpretation

The intended result is a three-example discovery panel in which every target family lies in the subcubic regime:


$$
\boxed{
\Lambda_m,\qquad
P_m^\downarrow,\qquad
P_m^{\updownarrow},
\qquad \Delta\le3.
}
$$


The workflow is

$$
\text{define family}
\rightarrow
\text{independent exact KnottedGraph dataset}
\rightarrow
\text{formula conjecture}
\rightarrow
\text{freeze}
\rightarrow
\text{distant held-out exact verification}.
$$

Until a mathematical derivation is supplied, an exact held-out success should be described as a **computer-generated conjectured closed form with exact held-out verification**, not as a proved theorem.